# VOCS data structure 

Variables, Objectives, Constraints, and other Settings (VOCS) helps define our optimization problems. 

In [1]:
from xopt.vocs import VOCS
from xopt.vocs import (
    get_objective_data,
    random_inputs,
    get_constraint_data,
    get_feasibility_data,
    normalize_inputs,
    denormalize_inputs,
)
from gest_api.vocs import MaximizeObjective
import pandas as pd
import numpy as np
import yaml

In [2]:
Y = """
variables:
  a: [0, 1e3] # Note that 1e3 usually parses as a str with YAML. 
  b: [-1, 1]
objectives:
  c: maximize
  d: minimize 
constraints:
  e: ['Less_than', 2]
  f: ['greater_than', 0]
constants:
  g: 123

"""

vocs = VOCS(**yaml.safe_load(Y))
vocs

VOCS(variables={'a': ContinuousVariable(dtype=None, default_value=None, domain=[0.0, 1000.0]), 'b': ContinuousVariable(dtype=None, default_value=None, domain=[-1.0, 1.0])}, objectives={'c': MaximizeObjective(dtype=None), 'd': MinimizeObjective(dtype=None)}, constraints={'e': LessThanConstraint(dtype=None, value=2.0), 'f': GreaterThanConstraint(dtype=None, value=0.0)}, constants={'g': Constant(dtype=None, value=123)}, observables={})

In [3]:
# as dict
dict(vocs)

{'variables': {'a': ContinuousVariable(dtype=None, default_value=None, domain=[0.0, 1000.0]),
  'b': ContinuousVariable(dtype=None, default_value=None, domain=[-1.0, 1.0])},
 'objectives': {'c': MaximizeObjective(dtype=None),
  'd': MinimizeObjective(dtype=None)},
 'constraints': {'e': LessThanConstraint(dtype=None, value=2.0),
  'f': GreaterThanConstraint(dtype=None, value=0.0)},
 'constants': {'g': Constant(dtype=None, value=123)},
 'observables': {}}

In [4]:
#  re-parse dict
vocs2 = VOCS(**dict(vocs))

In [5]:
# Check that these are the same
vocs2 == vocs

True

In [6]:
# This replaces the old vocs["variables"]
getattr(vocs, "variables")

{'a': ContinuousVariable(dtype=None, default_value=None, domain=[0.0, 1000.0]),
 'b': ContinuousVariable(dtype=None, default_value=None, domain=[-1.0, 1.0])}

In [7]:
vocs.objectives["c"]

MaximizeObjective(dtype=None)

In [8]:
isinstance(vocs.objectives["c"], MaximizeObjective)

True

In [9]:
# json
vocs.model_dump_json()

'{"variables":{"a":{"dtype":null,"default_value":null,"domain":[0.0,1000.0],"type":"ContinuousVariable"},"b":{"dtype":null,"default_value":null,"domain":[-1.0,1.0],"type":"ContinuousVariable"}},"objectives":{"c":{"dtype":null,"type":"MaximizeObjective"},"d":{"dtype":null,"type":"MinimizeObjective"}},"constraints":{"e":{"dtype":null,"value":2.0,"type":"LessThanConstraint"},"f":{"dtype":null,"value":0.0,"type":"GreaterThanConstraint"}},"constants":{"g":{"dtype":null,"value":123,"type":"Constant"}},"observables":{}}'

# Objective Evaluation

In [10]:
data = pd.DataFrame(random_inputs(vocs, 10))
# Add some outputs
data["c"] = data["a"] + data["b"]
data["d"] = data["a"] - data["b"]
data["e"] = data["a"] * 2 + data["b"] * 2
data["f"] = data["a"] * 2 - data["b"] * 2
data.index = np.arange(len(data)) + 5  # custom index
data

,a,b,g,c,d,e,f
5,624.356195,0.620891,123,624.977087,623.735304,1249.954173,1247.470608
6,878.533394,0.923303,123,879.456697,877.610092,1758.913394,1755.220183
7,990.977531,-0.868129,123,990.109402,991.845660,1980.218804,1983.691321
8,911.045694,-0.885217,123,910.160476,911.930911,1820.320953,1823.861823
9,69.307684,-0.280973,123,69.026711,69.588656,138.053422,139.177312
10,558.088791,0.385899,123,558.474690,557.702893,1116.949380,1115.405785
11,685.674892,-0.692350,123,684.982542,686.367242,1369.965084,1372.734484
12,689.892765,0.373324,123,690.266089,689.519442,1380.532178,1379.038884
13,412.437040,0.704209,123,413.141249,411.732831,826.282498,823.465662
14,69.026948,-0.954961,123,68.071987,69.981909,136.143974,139.963818


In [11]:
# These are in standard form for minimization, available as a function
get_objective_data(vocs, data)

,objective_c,objective_d
5,-624.977087,623.735304
6,-879.456697,877.610092
7,-990.109402,991.845660
8,-910.160476,911.930911
9,-69.026711,69.588656
10,-558.474690,557.702893
11,-684.982542,686.367242
12,-690.266089,689.519442
13,-413.141249,411.732831
14,-68.071987,69.981909


In [12]:
# use the to_numpy() method to convert for low level use.
get_objective_data(vocs, data).to_numpy()

array([[-624.97708671,  623.73530378],
       [-879.45669716,  877.61009166],
       [-990.10940216,  991.84566036],
       [-910.16047647,  911.93091138],
       [ -69.02671095,   69.58865618],
       [-558.4746902 ,  557.70289261],
       [-684.98254221,  686.36724195],
       [-690.26608911,  689.51944189],
       [-413.14124881,  411.73283116],
       [ -68.07198685,   69.98190876]])

In [13]:
get_constraint_data(vocs, data)

,constraint_e,constraint_f
5,1247.954173,-1247.470608
6,1756.913394,-1755.220183
7,1978.218804,-1983.691321
8,1818.320953,-1823.861823
9,136.053422,-139.177312
10,1114.949380,-1115.405785
11,1367.965084,-1372.734484
12,1378.532178,-1379.038884
13,824.282498,-823.465662
14,134.143974,-139.963818


In [14]:
get_feasibility_data(vocs, data)

,feasible_e,feasible_f,feasible
5,False,True,False
6,False,True,False
7,False,True,False
8,False,True,False
9,False,True,False
10,False,True,False
11,False,True,False
12,False,True,False
13,False,True,False
14,False,True,False


In [15]:
# normalize inputs to unit domain [0,1]
normed_data = normalize_inputs(vocs, data)
normed_data

,a,b
5,0.624356,0.810446
6,0.878533,0.961651
7,0.990978,0.065935
8,0.911046,0.057391
9,0.069308,0.359514
10,0.558089,0.692949
11,0.685675,0.153825
12,0.689893,0.686662
13,0.412437,0.852104
14,0.069027,0.022520


In [16]:
# and denormalize
denormalize_inputs(vocs, normed_data)

,a,b
5,624.356195,0.620891
6,878.533394,0.923303
7,990.977531,-0.868129
8,911.045694,-0.885217
9,69.307684,-0.280973
10,558.088791,0.385899
11,685.674892,-0.692350
12,689.892765,0.373324
13,412.437040,0.704209
14,69.026948,-0.954961


# Error handling

In [17]:
Y = """
variables:
  a: [0, 1e3] # Note that 1e3 usually parses as a str with YAML. 
  b: [-1, 1]
objectives:
  c: maximize
  d: minimize 
constraints:
  e: ['Less_than', 2]
  f: ['greater_than', 0]
constants:
  g: 1234

"""

vocs = VOCS(**yaml.safe_load(Y))

In [18]:
d = {"a": [1, 2, 3]}

df = pd.DataFrame(d)
df2 = pd.DataFrame(df).copy()

df2["b"] = np.nan
df2["b"] - 1

0   NaN
1   NaN
2   NaN
Name: b, dtype: float64

In [19]:
data["a"] = np.nan

In [20]:
a = 2


def f(x=a):
    return x


a = 99
f()

2

In [21]:
pd.DataFrame(6e66, index=[1, 2, 3], columns=["A"])

,A
1,6.000000e+66
2,6.000000e+66
3,6.000000e+66


In [22]:
# These are in standard form for minimization

data = pd.DataFrame({"c": [1, 2, 3, 4]}, index=[9, 3, 4, 5])

get_objective_data(vocs, data)

,objective_c,objective_d
9,-1.0,inf
3,-2.0,inf
4,-3.0,inf
5,-4.0,inf
